# CondAptNet — Stage 1 Training (Colab / T4)

**Before running:** Set runtime to GPU (Runtime → Change runtime type → T4 GPU).  
**Steps:** GPU check → clone repo → download data → resume check → train → evaluate.

In [1]:
# Cell 1 — GPU check
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → GPU (T4)."
)

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024 ** 3
print(f"GPU  : {gpu.name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
assert vram_gb >= 12, f"Expected ≥12 GB VRAM, got {vram_gb:.1f} GB"
print("GPU check passed.")

GPU  : Tesla T4
VRAM : 14.6 GB
CUDA : 12.8
GPU check passed.


In [2]:
# Cell 2 — Clone repo and install dependencies
import subprocess, sys

result = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/shivanshb828/CondAptNet.git"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Repo cloned.")
else:
    # Already cloned — pull latest
    subprocess.run(["git", "-C", "CondAptNet", "pull", "--quiet"],
                   check=True)
    print("Repo already present — pulled latest.")

import os
os.chdir("CondAptNet")
print(f"Working directory: {os.getcwd()}")

packages = [
    "fair-esm",
    "ViennaRNA",
    "scikit-learn",
    "scipy",
    "biopython",
    "gdown",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
    check=True
)
print("Dependencies installed.")

Repo cloned.
Working directory: /content/CondAptNet
Dependencies installed.


In [3]:
# Cell 3 — Download data from Google Drive
# Replace the IDs below with your actual Google Drive file/folder IDs.
# master_dataset.csv  → share as Anyone with link, copy the file ID
# vienna_cache.pkl    → same
# protein_embeddings  → share the folder, copy the folder ID

import os
import gdown

MASTER_DATASET_ID      = "1LlVsXnMkrEZm-JlkVQv8Bwffxk8hueqB"
VIENNA_CACHE_ID        = "1mRaGNcVsaSb7QywORqCHSIMQWcHXpvV2"
PROTEIN_EMBEDDINGS_ID  = "1O_y08uU7Gs14LIKBCZ02L1QqRIItmvuQ"

os.makedirs("data/processed/protein_embeddings", exist_ok=True)
os.makedirs("models/checkpoints/pretrain",       exist_ok=True)

print("Downloading master_dataset.csv ...")
gdown.download(
    f"https://drive.google.com/uc?id={MASTER_DATASET_ID}",
    "data/processed/master_dataset.csv",
    quiet=False,
)

print("Downloading vienna_cache.pkl ...")
gdown.download(
    f"https://drive.google.com/uc?id={VIENNA_CACHE_ID}",
    "data/processed/vienna_cache.pkl",
    quiet=False,
)

print("Downloading protein_embeddings folder ...")
gdown.download_folder(
    f"https://drive.google.com/drive/folders/{PROTEIN_EMBEDDINGS_ID}",
    output="data/processed/protein_embeddings",
    quiet=False,
    use_cookies=False,
)

# Verify
import pandas as pd
df = pd.read_csv("data/processed/master_dataset.csv")
ready = (
    df["sequence"].notna() &
    (df["needs_sequence_enrichment"] == False) &
    df["protein_sequence"].notna()
).sum()
n_emb = len(os.listdir("data/processed/protein_embeddings"))
print(f"master_dataset.csv : {len(df):,} rows  ({ready:,} training-ready)")
print(f"protein_embeddings : {n_emb} .npy files")
print("Data download complete.")

Downloading...
From: https://drive.google.com/uc?id=1LlVsXnMkrEZm-JlkVQv8Bwffxk8hueqB
To: /content/CondAptNet/data/processed/master_dataset.csv
100%|██████████| 1.98M/1.98M [00:00<00:00, 85.3MB/s]


Downloading...
From: https://drive.google.com/uc?id=1mRaGNcVsaSb7QywORqCHSIMQWcHXpvV2
To: /content/CondAptNet/data/processed/vienna_cache.pkl
100%|██████████| 1.13M/1.13M [00:00<00:00, 148MB/s]
Retrieving folder contents


Processing file 1sDfDabUKpcPo6P_8qrF3Pgq6Suu3rd_U 5d64efa33ef955a9286fdf2d27b5acc6.npy
Processing file 1WwNfXMDjRKFBz6QRDOyziU4UNjwZZ4Cm 7aa7242ed3dde05e49c7e4c740e28b97.npy
Processing file 1SmFPovpDiHKaFTzRY19CVJly_Ndtf2yW 7ff608cd5e7eebd672fef443f43e0c9e.npy
Processing file 1CBQVxAkgMgBL-xbzK4jKKmEaa7PfuYxM 54ce9b483c09461b4338cf133f87aeef.npy
Processing file 1tINfiCJK3XDoZylAaY8-dCsQPcCyOM7Y 56a97a213d4c9ec0be992e3b87d4a4d5.npy
Processing file 1lr4mLtR_xALUj4RsYskpb65HsKkurlV9 61b510e6b4e22446b5c80b79268978c9.npy
Processing file 1yVOva_tH2toxQnPOYjUjFVxo3H5yW_9E 96b7ba11d33c2c9a4d00a5186b536604.npy
Processing file 15ySu6vQmV5EPu-QubhYEJ9a8WwmlTI9u 98fa37659747c1072800599b8f609fff.npy
Processing file 1v96t3qS5sOnJNB0JlXGWp4B5Kzg7njRO 757130cd50a3b189138201447917fbaf.npy
Processing file 1-JzEenV9wLiBXO0CgIRzHDPcna0KzFkH a073f972c1685eab2a6639c66978092a.npy
Processing file 1fYE_lwwpBf1b8JkmX4HlRDgfwW-yZq1s a510eadb0df840e958233f484781dde7.npy
Processing file 1HK0-On1oNqpuVnYtqBsIySndPW

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1sDfDabUKpcPo6P_8qrF3Pgq6Suu3rd_U
To: /content/CondAptNet/data/processed/protein_embeddings/5d64efa33ef955a9286fdf2d27b5acc6.npy
100%|██████████| 1.97M/1.97M [00:00<00:00, 162MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WwNfXMDjRKFBz6QRDOyziU4UNjwZZ4Cm
To: /content/CondAptNet/data/processed/protein_embeddings/7aa7242ed3dde05e49c7e4c740e28b97.npy
100%|██████████| 1.97M/1.97M [00:00<00:00, 67.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SmFPovpDiHKaFTzRY19CVJly_Ndtf2yW
To: /content/CondAptNet/data/processed/protein_embeddings/7ff608cd5e7eebd672fef443f43e0c9e.npy
100%|██████████| 1.80M/1.80M [00:00<00:00, 199MB/s]
Downloading...
From: https://drive.google.com/uc?id=1CBQVxAkgMgBL-xbzK4jKKmEaa7PfuYxM
To: /content/CondAptNet/data/processed/protein_embeddings/54ce9b483c09461b4338cf133f87aeef.npy
100%|█████████

master_dataset.csv : 3,821 rows  (2,364 training-ready)
protein_embeddings : 23 .npy files
Data download complete.


In [4]:
# Cell 4 — Resume logic
# Finds the latest epoch checkpoint so training can continue after a disconnect.

import glob

CHECKPOINT_DIR = "models/checkpoints/pretrain"

existing = sorted(glob.glob(f"{CHECKPOINT_DIR}/epoch_*.pt"))

if existing:
    latest = existing[-1]
    import torch
    meta = torch.load(latest, map_location="cpu")
    last_epoch    = meta.get("epoch", "?")
    best_val_mcc  = meta.get("best_val_mcc", meta.get("val_mcc", float("nan")))
    patience      = meta.get("patience_count", "?")
    RESUME_FLAG   = "--resume"
    print(f"Checkpoint found  : {latest}")
    print(f"Last epoch        : {last_epoch}")
    print(f"Best val MCC      : {best_val_mcc:.4f}")
    print(f"Patience count    : {patience}")
    print("Will resume training from next epoch.")
else:
    RESUME_FLAG = ""
    print("No checkpoint found — starting fresh.")

No checkpoint found — starting fresh.


In [5]:
# Cell 5 — Train (T4 settings: batch_size=16, max_prot_len=128)
import subprocess, os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "python", "scripts/training/train.py",
    "--max-epochs",    "100",
    "--batch-size",    "16",
    "--max-prot-len",  "128",
    "--checkpoint-dir", CHECKPOINT_DIR,
]
if RESUME_FLAG:
    cmd.append(RESUME_FLAG)

env = os.environ.copy()
env["PYTHONUNBUFFERED"]    = "1"
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("-" * 60)
print(f"Training finished (exit code {proc.returncode}).")

Command: python scripts/training/train.py --max-epochs 100 --batch-size 16 --max-prot-len 128 --checkpoint-dir models/checkpoints/pretrain
------------------------------------------------------------
[CondAptNet] Device: cuda
2026-05-11 15:51:13,142 INFO Device confirmed at runtime: cuda
2026-05-11 15:51:13,145 INFO Loading data from /content/CondAptNet/data/processed/master_dataset.csv
2026-05-11 15:51:13,183 INFO Training-ready rows: 2364 / 3821 total
2026-05-11 15:51:13,188 INFO Split: 1785 train / 297 val / 282 test rows  (176 / 37 / 39 families)
2026-05-11 15:51:13,195 INFO Vienna cache: 6330 entries
2026-05-11 15:51:13,195 INFO Building CondAptNet...
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D-contact-regression.pt
2026-

In [ ]:
# Cell 6 — Evaluate best checkpoint
import subprocess, os

best_ckpt = os.path.join(CHECKPOINT_DIR, "best.pt")

if not os.path.exists(best_ckpt):
    print(f"No best checkpoint at {best_ckpt} — run Cell 5 first.")
else:
    for split in ("val", "test"):
        print(f"\n{'='*55}")
        print(f"Evaluating on {split.upper()} split")
        print(f"{'='*55}")
        result = subprocess.run(
            [
                "python", "scripts/evaluation/evaluate.py",
                "--checkpoint", best_ckpt,
                "--split",      split,
            ],
            capture_output=False, text=True,
        )